# XGBoost Encoding Benchmark – Train Only (Memory-Efficient)

Loads **training data only** to avoid memory pressure when running full data.
Models and encoders are saved to `models/` for later test evaluation.

| Step | Description |
|------|-------------|
| 0 | Set DEBUG flag – `0` = full data, `1` = 10K rows, `2` = 10% sample |
| 1 | (First run only) Generate `config/level_mapping_reference.csv` |
| 2 | Load **train data only** |
| 2.5 | Quick Iterative Experiment – tune hyperparameters, see train lift chart |
| 3–6 | Run all 4 encoding experiments + train XGBoost |
| 7 | Compare train metrics across encodings |

**Next step:** Run `main_test_eval.ipynb` to evaluate on test data.

---
## 0 · Setup & DEBUG flag

In [ ]:
import os, sys

PROJECT_ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, os.path.join(PROJECT_ROOT, "code"))

# ── Toggle this flag ─────────────────────────────────────────────────────────
DEBUG = 2   # 0 = full data  |  1 = 10K rows (fast smoke test)  |  2 = 10% sample (medium)
# ─────────────────────────────────────────────────────────────────────────────

_label = {0: 'FULL DATA', 1: '10K rows', 2: '10% sample'}.get(DEBUG, 'FULL DATA')
print(f"Project root : {PROJECT_ROOT}")
print(f"DEBUG        : {DEBUG}  ({_label})")

Project root : /Users/Mach/dev/aps/code/26CF_Dmod_v1/experiment_levels
DEBUG        : 1  (10K rows)


---
## 1 · Generate level mapping CSV (run once)

Skip this cell if `config/level_mapping_reference.csv` already exists.

In [2]:
import importlib, create_level_mapping as clm

MAPPING_CSV = os.path.join(PROJECT_ROOT, "config", "level_mapping_reference.csv")

if not os.path.exists(MAPPING_CSV):
    print("Mapping CSV not found – generating now ...")
    clm.DEBUG = DEBUG
    clm.main()
else:
    print(f"✅ Mapping CSV already exists:\n   {MAPPING_CSV}")
    print("   Delete it and re-run this cell to regenerate.")

✅ Mapping CSV already exists:
   /Users/Mach/dev/aps/code/26CF_Dmod_v1/experiment_levels/config/level_mapping_reference.csv
   Delete it and re-run this cell to regenerate.


---
## 2 · Load training data only

Uses `load_train_only()` — loads only the training parquet.
Memory is ~50% of loading both datasets.

| DEBUG | Data loaded |
|-------|-------------|
| 0 | Full training set |
| 1 | First 10K rows |
| 2 | Random 10% sample (medium-scale — good for ~1 hr runs) |

In [ ]:
from encoding_strategies import load_train_only, get_y

train_raw = load_train_only(debug=DEBUG)

y_train = get_y(train_raw)

print(f"y_train : mean={y_train.mean():.4f}  std={y_train.std():.4f}")
print(f"Memory  : {train_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")

[DEBUG-10K] Loading train ...


---
## 2.5 · Quick Iterative Experiment

**Iterative tuning workflow:**
1. Edit `config.json` (change `learning_rate`, `n_estimators`, etc.)
2. Run this cell
3. See **train-set** lift chart + metrics immediately
4. Repeat — no kernel restart needed

Set `ITER_ENC` to choose which encoding to benchmark:
- `1` = Type 1 Ordinal
- `2` = Type 2 Binary
- `3` = Type 3 Actuarial
- `4` = Type 4 Custom

In [ ]:
# ── Edit these two lines, then run ───────────────────────────────────────────
ITER_ENC  = 1          # Encoding type: 1 / 2 / 3 / 4
ITER_NAME = 'run_v1'   # Label for this run (shows in lift chart title)
# ─────────────────────────────────────────────────────────────────────────────

import model_training as _mt
_mt.reload_config()   # Re-read config.json — no kernel restart needed

from encoding_strategies import encode_type1_ordinal, encode_type2_binary
from encoding_strategies import encode_type3_actuarial, encode_type4_custom
from model_training import train_on_train_data, save_experiment
from visualization import lift_chart

_enc_map = {
    1: lambda: encode_type1_ordinal(train_raw),
    2: lambda: encode_type2_binary(train_raw),
    3: lambda: encode_type3_actuarial(train_raw),
    4: lambda: encode_type4_custom(train_raw),
}
_enc_names = {1:'type1_ordinal', 2:'type2_binary', 3:'type3_actuarial', 4:'type4_custom'}

if ITER_ENC not in _enc_map:
    raise ValueError(f'ITER_ENC must be 1/2/3/4, got {ITER_ENC}')

_out = _enc_map[ITER_ENC]()
_X_tr, _feats, _encoders = _out[0], _out[1], _out[2]

_exp_name = f"{_enc_names[ITER_ENC]}_{ITER_NAME}"
_result = train_on_train_data(
    experiment_name=_exp_name,
    X_train=_X_tr, y_train=y_train,
    feature_names=_feats,
)

# ✅ Train-set lift chart (instant feedback)
lift_chart(_result, test_df=train_raw, bins=10, print_table=True,
           title=f'TRAIN Lift — {_enc_names[ITER_ENC]} | {ITER_NAME} | RMSE={_result["metrics"]["train_rmse"]:.2f}')

print(f'  Train RMSE : {_result["metrics"]["train_rmse"]:.4f}')
print(f'  Train MAE  : {_result["metrics"]["train_mae"]:.4f}')
print(f'  Train R²   : {_result["metrics"]["train_r2"]:.4f}')

# Save model + encoders for later test evaluation
save_experiment(_result, encoders=_encoders)

---
## 3 · Experiment 1 – Ordinal encoding (Type 1)

In [ ]:
from encoding_strategies import encode_type1_ordinal
from model_training import train_on_train_data, save_experiment
from visualization import lift_chart

X_tr1, feats1, enc1 = encode_type1_ordinal(train_raw)

result1 = train_on_train_data(
    experiment_name='type1_ordinal',
    X_train=X_tr1, y_train=y_train,
    feature_names=feats1,
)

lift_chart(result1, test_df=train_raw, bins=10, print_table=False,
           title=f'TRAIN Lift — Type 1 Ordinal | RMSE={result1["metrics"]["train_rmse"]:.2f}')

save_experiment(result1, encoders=enc1)

---
## 4 · Experiment 2 – Binary encoding (Type 2)

In [ ]:
from encoding_strategies import encode_type2_binary

X_tr2, feats2, enc2 = encode_type2_binary(train_raw)

result2 = train_on_train_data(
    experiment_name='type2_binary',
    X_train=X_tr2, y_train=y_train,
    feature_names=feats2,
)

lift_chart(result2, test_df=train_raw, bins=10, print_table=False,
           title=f'TRAIN Lift — Type 2 Binary | RMSE={result2["metrics"]["train_rmse"]:.2f}')

save_experiment(result2, encoders=enc2)

---
## 5 · Experiment 3 – Actuarial encoding (Type 3)

In [ ]:
from encoding_strategies import encode_type3_actuarial

X_tr3, feats3, enc3 = encode_type3_actuarial(train_raw)

result3 = train_on_train_data(
    experiment_name='type3_actuarial',
    X_train=X_tr3, y_train=y_train,
    feature_names=feats3,
)

lift_chart(result3, test_df=train_raw, bins=10, print_table=False,
           title=f'TRAIN Lift — Type 3 Actuarial | RMSE={result3["metrics"]["train_rmse"]:.2f}')

save_experiment(result3, encoders=enc3)

---
## 6 · Experiment 4 – Custom encoding (Type 4)

In [ ]:
from encoding_strategies import encode_type4_custom

X_tr4, feats4, enc4 = encode_type4_custom(train_raw)

result4 = train_on_train_data(
    experiment_name='type4_custom',
    X_train=X_tr4, y_train=y_train,
    feature_names=feats4,
)

lift_chart(result4, test_df=train_raw, bins=10, print_table=False,
           title=f'TRAIN Lift — Type 4 Custom | RMSE={result4["metrics"]["train_rmse"]:.2f}')

save_experiment(result4, encoders=enc4)

---
## 7 · Compare train metrics

⚠️ These are **training-set** metrics (optimistic). Run `main_test_eval.ipynb` for true test-set comparison.

In [ ]:
import pandas as pd

_all_results = [result1, result2, result3, result4]

rows = []
for r in _all_results:
    rows.append({
        'experiment':   r['experiment_name'],
        'train_rmse':   r['metrics']['train_rmse'],
        'train_mae':    r['metrics']['train_mae'],
        'train_r2':     r['metrics']['train_r2'],
        'train_time_s': r['metrics']['training_time_sec'],
        'n_features':   r['metrics']['n_features'],
    })

summary_tr = pd.DataFrame(rows).set_index('experiment').sort_values('train_rmse')
print("\n TRAIN-SET METRICS (sorted by RMSE — lower is better)")
print("=" * 70)
print(summary_tr.to_string())
print("\n⚠️  Run main_test_eval.ipynb for test-set comparison")